# Tema 9 — Fundamentos y aplicaciones de la visión computacional

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecamposv/nlp-vision/blob/main/semana-05/Tema-09/Tema_9.ipynb)

Puedes ejecutar este notebook localmente (VS Code / Jupyter) o en **Google Colab** dando clic en el badge de arriba.

> **Requisitos:** `opencv-python`, `numpy`, `matplotlib`. Coloca la imagen `Tigre.png` (o `Tigre.jpg`) en el mismo directorio del notebook.


## 1. Ver canales en OpenCV-Python

La siguiente celda **carga una imagen** desde disco usando `cv2.imread`, que en OpenCV devuelve los píxeles en orden **BGR** (azul, verde, rojo) y no RGB. Después:

- Imprime `img.shape`, que devuelve la tupla `(alto, ancho, 3)` para imágenes a color.
- Usa `cv2.split` para **separar los tres canales** en tres matrices 2D independientes (`b`, `g`, `r`), cada una con forma `(H, W)`.

Esto permite ver cómo está organizada internamente una imagen en color y trabajar canal por canal.


In [ ]:
import cv2
img_bgr = cv2.imread("Tigre.jpg")          # Carga en BGR (ver 2.3)
print(img_bgr.shape)                      # (H, W, 3) en color
b, g, r = cv2.split(img_bgr)              # Extraer canales
print(b.shape, g.shape, r.shape)          # Cada uno (H, W)

## 2. Convertir BGR a RGB

OpenCV trabaja en **BGR**, pero `matplotlib` y la mayoría de librerías de visualización esperan **RGB**. Si mostramos directamente la imagen sin convertirla, los colores aparecen invertidos (lo rojo se ve azul y viceversa).

La celda usa `cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)` para **reordenar los canales** y obtener una versión `img_rgb` lista para visualizar o guardar con librerías que asumen RGB.


In [ ]:
# Conversión correcta para visualizar con matplotlib o guardar
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

## 3. Funciones básicas de preprocesamiento

La celda encadena cuatro operaciones fundamentales de preprocesamiento que se usan típicamente como paso previo a un modelo de visión:

1. **`cv2.cvtColor(..., COLOR_BGR2GRAY)`** — convierte la imagen a escala de grises (1 canal). Reduce datos y aísla la información de intensidad.
2. **`cv2.equalizeHist(gray)`** — aplica **ecualización de histograma** para mejorar el contraste redistribuyendo los niveles de gris.
3. **`cv2.GaussianBlur(eq, (5,5), 1.0)`** — **suaviza** la imagen con un kernel gaussiano de 5×5 y σ=1.0. Reduce ruido antes de detectar bordes.
4. **`cv2.Canny(blur, 100, 200)`** — detecta **bordes** con el algoritmo de Canny usando umbrales inferior y superior de 100 y 200.

El orden importa: gris → ecualizar → suavizar → bordes.


In [ ]:
gray   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)         # a grises
eq     = cv2.equalizeHist(gray)                            #ecualización(contraste)
blur   = cv2.GaussianBlur(eq, (5,5), 1.0)                  # suavizado
edges  = cv2.Canny(blur, 100, 200)                         # bordes

## 4. Pipeline completo de preprocesamiento

Esta celda integra **todo el flujo de procesamiento** sobre la imagen `Tigre.png` y guarda cada resultado en la carpeta `salidas/` (se crea con `os.makedirs(..., exist_ok=True)`).

**Bloques numerados dentro de la celda:**

1. **Carga (BGR)** — `cv2.imread`, valida que el archivo exista.
2. **BGR → RGB** — conversión correcta para visualización.
3. **Espacios de color** — convierte a `GRAY`, `HSV`, `Lab` y `YCrCb` (útiles para análisis y segmentación).
4. **Separación de canales** — extrae los canales individuales de cada espacio (R, G, B / H, S, V / L, a, b / Y, Cr, Cb).
5. **Histogramas** — la función auxiliar `guardar_histograma` grafica la distribución de intensidades por canal con `matplotlib`.
6. **Ecualización** — mejora el contraste en escala de grises y también sobre la **luminancia Y** del espacio YCrCb (forma correcta de ecualizar imágenes a color sin alterar la cromaticidad).
7. **Operaciones geométricas** — `cv2.resize` (512×512), recorte central al 60 % y `cv2.warpAffine` con `getRotationMatrix2D` para rotar 15°.
8. **Suavizado + bordes** — `GaussianBlur` seguido de `Canny`.
9. **Segmentación en HSV** — define un rango de **naranja** `(H 5–25, S/V ≥ 50)` con `cv2.inRange`, crea una máscara binaria y aplica `cv2.bitwise_and` para aislar el pelaje del tigre.

Al final imprime la ruta absoluta de la carpeta `salidas/` con todos los archivos generados.


In [2]:
# Script simple: Tema 9 - Fundamentos y aplicaciones de la visión computacional
# Requisitos: opencv-python, numpy, matplotlib
# Coloca "Tigre.png" en el mismo directorio o la nombre_imagen.png

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

OUT_DIR = "salidas"
os.makedirs(OUT_DIR, exist_ok=True)

def guardar_histograma(data, titulo, nombre_png, bins=256, rango=(0,255)):
    plt.figure(figsize=(6,4))
    plt.hist(data.ravel(), bins=bins, range=rango)
    plt.title(titulo)
    plt.xlabel("Intensidad / Valor")
    plt.ylabel("Frecuencia")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, nombre_png), dpi=160)
    plt.close()

# 1) Carga (BGR por defecto)
img_bgr = cv2.imread("Tigre.png")
if img_bgr is None:
    raise FileNotFoundError("No se encontró 'Tigre.png' en el directorio actual.")
cv2.imwrite(os.path.join(OUT_DIR, "01_original_BGR.png"), img_bgr)

# 2) Conversión para visualización correcta (BGR -> RGB)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# Guardamos una versión “RGB” (convirtiendo de vuelta a BGR para imwrite)
cv2.imwrite(os.path.join(OUT_DIR, "02_RGB_correcto.png"), cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))

# 3) Espacios de color principales
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
img_hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
img_lab  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2Lab)
img_ycc  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)

# Para “ver” HSV/Lab/YCrCb como imagen, conviértelos a BGR, se guardan en la carpeta de salida:
cv2.imwrite(os.path.join(OUT_DIR, "03_gray.png"), img_gray)
cv2.imwrite(os.path.join(OUT_DIR, "04_hsv_vis.png"),  cv2.cvtColor(img_hsv, cv2.COLOR_HSV2BGR))
cv2.imwrite(os.path.join(OUT_DIR, "05_lab_vis.png"),  cv2.cvtColor(img_lab, cv2.COLOR_Lab2BGR))
cv2.imwrite(os.path.join(OUT_DIR, "06_ycrcb_vis.png"),cv2.cvtColor(img_ycc, cv2.COLOR_YCrCb2BGR))

# 4) Canales por espacio
# RGB (a partir de BGR)
b, g, r = cv2.split(img_bgr)
cv2.imwrite(os.path.join(OUT_DIR, "07_canal_R.png"), r)
cv2.imwrite(os.path.join(OUT_DIR, "08_canal_G.png"), g)
cv2.imwrite(os.path.join(OUT_DIR, "09_canal_B.png"), b)

# HSV
H, S, V = cv2.split(img_hsv)
cv2.imwrite(os.path.join(OUT_DIR, "10_canal_H.png"), H)
cv2.imwrite(os.path.join(OUT_DIR, "11_canal_S.png"), S)
cv2.imwrite(os.path.join(OUT_DIR, "12_canal_V.png"), V)

# Lab
L, a, b_lab = cv2.split(img_lab)
cv2.imwrite(os.path.join(OUT_DIR, "13_canal_L.png"), L)
cv2.imwrite(os.path.join(OUT_DIR, "14_canal_a.png"), a)
cv2.imwrite(os.path.join(OUT_DIR, "15_canal_b.png"), b_lab)

# YCrCb
Y, Cr, Cb = cv2.split(img_ycc)
cv2.imwrite(os.path.join(OUT_DIR, "16_canal_Y.png"), Y)
cv2.imwrite(os.path.join(OUT_DIR, "17_canal_Cr.png"), Cr)
cv2.imwrite(os.path.join(OUT_DIR, "18_canal_Cb.png"), Cb)

# 5) Histogramas (RGB por canal, GRAY, HSV, Y)
guardar_histograma(r, "Histograma canal R (RGB)", "19_hist_R.png")
guardar_histograma(g, "Histograma canal G (RGB)", "20_hist_G.png")
guardar_histograma(b, "Histograma canal B (RGB)", "21_hist_B.png")
guardar_histograma(img_gray, "Histograma (grises)", "22_hist_grises.png")
guardar_histograma(H, "Histograma canal H (HSV)", "23_hist_H.png", bins=180, rango=(0,179))
guardar_histograma(S, "Histograma canal S (HSV)", "24_hist_S.png")
guardar_histograma(V, "Histograma canal V (HSV)", "25_hist_V.png")
guardar_histograma(Y, "Histograma luminancia Y (YCrCb)", "26_hist_Y.png")

# 6) Ecualización (mejora contraste)
gray_eq = cv2.equalizeHist(img_gray)  # en grises
ycc_eq  = img_ycc.copy()              # en luminancia Y
ycc_eq[:, :, 0] = cv2.equalizeHist(ycc_eq[:, :, 0])
eq_bgr = cv2.cvtColor(ycc_eq, cv2.COLOR_YCrCb2BGR)
cv2.imwrite(os.path.join(OUT_DIR, "27_gray_ecualizada.png"), gray_eq)
cv2.imwrite(os.path.join(OUT_DIR, "28_bgr_ecualizadaY.png"), eq_bgr)

# 7) Operaciones básicas: redimensionar, recorte central y rotación
resized = cv2.resize(img_bgr, (512, 512), interpolation=cv2.INTER_AREA)
h, w = img_bgr.shape[:2]
frac = 0.6
ch, cw = int(h*frac), int(w*frac)
y1, x1 = (h - ch)//2, (w - cw)//2
crop = img_bgr[y1:y1+ch, x1:x1+cw]
M = cv2.getRotationMatrix2D((w/2.0, h/2.0), 15, 1.0)
rot = cv2.warpAffine(img_bgr, M, (w, h))
cv2.imwrite(os.path.join(OUT_DIR, "29_redimension_512.png"), resized)
cv2.imwrite(os.path.join(OUT_DIR, "30_recorte_central.png"), crop)
cv2.imwrite(os.path.join(OUT_DIR, "31_rotacion_15.png"), rot)

# 8) Suavizado + bordes (Canny)
blur = cv2.GaussianBlur(gray_eq, (5,5), 1.0)
edges = cv2.Canny(blur, 100, 200)
cv2.imwrite(os.path.join(OUT_DIR, "32_blur_gauss.png"), blur)
cv2.imwrite(os.path.join(OUT_DIR, "33_bordes_canny.png"), edges)

# 9) Segmentación en HSV (ejemplo: “naranja” del pelaje)
# Rango aproximado de naranja en OpenCV (H en 0–179)
LOW  = (5,  50,  50)   # (H, S, V)
HIGH = (25, 255, 255)
mask = cv2.inRange(img_hsv, LOW, HIGH)
segm = cv2.bitwise_and(img_bgr, img_bgr, mask=mask)
cv2.imwrite(os.path.join(OUT_DIR, "34_mask_naranja.png"), mask)
cv2.imwrite(os.path.join(OUT_DIR, "35_segmentacion_naranja.png"), segm)

print("[OK] Proceso completado. Revisa la carpeta:", os.path.abspath(OUT_DIR))

[OK] Proceso completado. Revisa la carpeta: /content/salidas
